# Regime-classifier audit

This audit addresses regime support in training/test windows, the 2023 SVB event study, and 2022 sequential out-of-sample evidence. It reads source files without changing them. Put this notebook and `regime_audit_helpers.py` in the repository's `audit/` directory, or change ROOT below.

The rule is a majority of three votes: BTC bearish, ETH bearish, and Fear & Greed <40. A coin is bearish when close<SMA50, SMA20<SMA50, and simple-moving-average RSI14<50. ADX and volatility are diagnostic variables, not votes. This is a deterministic regime rule, not a trained classifier.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'dataset' / 'prices.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project.')
for folder in [ROOT / 'depeg_labeller', ROOT / 'regime_classifier', ROOT / 'logit']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from regime_audit_helpers import load_panel, summary, check_predictions, FEATURES

OUT = ROOT / 'regime_classifier/outputs/audit'
OUT.mkdir(parents=True, exist_ok=True)
regime, labels, panel = load_panel(ROOT)
print('Analysis uses predictor dates t; labels refer to event dates t+1.')
display(panel.groupby(['window', 'coin']).agg(
    first_predictor_date=('date', 'min'), last_predictor_date=('date', 'max'),
    coin_days=('date', 'size')))


## Rule mechanics and information availability
Indicators at t use the t closing price and predict extremes at t+1. Verify that on-chain aggregates and sentiment were actually published by the forecast cutoff. A date stamp alone does not establish availability. No forward-looking windows occur in the supplied price-indicator formulas.

The supplied builder drops rows missing ADX even though ADX is not part of the rule. It uses simple rolling RSI/ADX rather than Wilder smoothing. Do not silently change those definitions during an audit. Record any later revisions and regenerate predictions. Missing dates can make rolling observation windows differ from calendar windows; initial NaN comparisons must not be treated as normal regimes.


In [ ]:
prices = pd.read_csv(ROOT / 'dataset/prices.csv', parse_dates=['date'])
sentiment = pd.read_csv(ROOT / 'dataset/sentiment.csv', parse_dates=['date'])
checks = []
for coin in ['BTC', 'ETH']:
    x = prices[prices.coin.eq(coin)].sort_values('date')
    checks.append({'coin': coin, 'rows': len(x),
                   'duplicate_dates': x.date.duplicated().sum(),
                   'non_daily_steps': x.date.diff().iloc[1:].ne(pd.Timedelta(days=1)).sum(),
                   'missing_or_nonfinite_close': (~np.isfinite(x.close)).sum()})
display(pd.DataFrame(checks))
print('Duplicate sentiment dates:', sentiment.date.duplicated().sum())
expected = ((regime.btc_sma_bearish + regime.btc_rsi_bearish).ge(2).astype(int)
            + (regime.eth_sma_bearish + regime.eth_rsi_bearish).ge(2).astype(int)
            + regime.fear_greed.lt(40).astype(int))
assert expected.eq(regime.stress_score).all()
assert expected.ge(2).astype(int).eq(regime.stress).all()
print('PASS: exported flags reproduce the stated majority rule.')
# Raw regime coverage and modelling coverage are different denominators.
display(regime.groupby(regime.date.dt.year).stress.agg(['count', 'sum', 'mean']))


## Training/test coverage, breaches, and episodes
Days count unique predictor dates; breaches count positive coin-days. Coin episodes are consecutive positive event-date runs within each coin, constructed before splitting windows. Missing dates break runs. Episode counts by regime describe overlap: one episode may appear in both regimes and must not be summed as independent episodes. Simultaneous coin episodes may reflect the same economic shock. Boundary episodes can be left/right censored by data coverage.

The 2022 training summary is the available 2022 modelling panel, not a claim that every 2023 expanding-window fold used only 2022. Test summaries below use the actual saved prediction dates. Inspect the model's fold logs for changing training windows.


In [ ]:
train = panel[panel.date.dt.year.eq(2022)].copy()
train['window'] = 'Available 2022 training panel'
pred_path = ROOT / 'logit/outputs/logit_decomposition/logit_oos_predictions.csv'
saved = pd.read_csv(pred_path, parse_dates=['date'])
verified = check_predictions(saved, panel)
display(verified.groupby('_merge', observed=True).size().rename('rows'))
label_mismatches = int((~verified.label_matches).sum())
regime_mismatches = int((~verified.regime_matches).sum())
print('Saved/current label mismatches:', label_mismatches)
print('Saved/current regime mismatches:', regime_mismatches)
verified.to_csv(OUT / 'saved_prediction_consistency.csv', index=False)
test_keys = saved.loc[saved.date.dt.year.eq(2023), ['coin', 'date']]
test = panel.merge(test_keys, on=['coin', 'date'], validate='one_to_one')
test['window'] = 'Actual 2023 test dates'
table = summary(pd.concat([train, test], ignore_index=True))
display(table)
table.to_csv(OUT / 'train_test_regime_support.csv', index=False)
for name, g in [('2022 training', train), ('2023 test', test)]:
    stressed = g.loc[g.stress.eq(1)]
    print(name, ':', g.date.nunique(), 'dates,', len(g), 'coin-days;',
          stressed.date.nunique(), 'stressed dates;',
          stressed.regime_spell_id.nunique(), 'stressed spells')
if test.loc[test.stress.eq(1), 'regime_spell_id'].nunique() <= 1:
    print('The 2023 test provides little evidence about general regime dependence: '
          'stressed observations occur in at most one market spell. Treat it as an SVB event study.')
fig, ax = plt.subplots(figsize=(12, 3))
for name, g in [('Training', train), ('Test', test)]:
    daily = g[['date', 'stress']].drop_duplicates().sort_values('date')
    ax.step(daily.date, daily.stress, where='post', label=name)
ax.set(yticks=[0, 1], yticklabels=['Normal', 'Stressed'], title='Model-sample regime coverage')
ax.legend(); fig.tight_layout(); fig.savefig(OUT / 'regime_coverage.png', dpi=160); plt.show(); plt.close(fig)


## 2023 SVB event study
Plot probabilities for outcomes dated t+1; the shaded flag is R(t), used by that forecast. A marker dated 11 March therefore comes from the 10 March forecast. The vertical line marks the event-study reference date of 10 March. Include both baseline M1, regime-intercept M2, and interaction M3 to distinguish adding the flag from allowing regime-dependent slopes.

Stale labels block the plots by default. After regenerating predictions, also verify model provenance: current input versions, train-only scaling/tuning, chronological folds, and a purge ensuring training outcomes predate the test period. Matching labels is necessary but does not prove predictions were regenerated. Historical predictions with stale labels should not be presented as predictions from the revised pipeline.


In [ ]:
if label_mismatches or regime_mismatches or not verified['_merge'].eq('both').all():
    print('EVENT STUDY BLOCKED: regenerate OOS predictions using the revised labels/regimes, then rerun.')
else:
    event = verified[verified.event_date.between('2023-03-01', '2023-03-20')].copy()
    for coin in ['USDC', 'DAI']:
        g = event[event.coin.eq(coin)].sort_values('event_date')
        if g.empty:
            print('No event-study predictions for', coin); continue
        fig, ax = plt.subplots(figsize=(11, 4))
        for col, title in [('p_m1_oos', 'M1 baseline'), ('p_m2_oos', 'M2 + regime')]:
            ax.plot(g.event_date, g[col], marker='.', label=title)
        for day in g.loc[g.stress.eq(1), 'event_date']:
            ax.axvspan(day - pd.Timedelta(hours=12), day + pd.Timedelta(hours=12),
                       color='gray', alpha=.2)
        hit = g.depeg_dynamic.eq(1)
        ax.scatter(g.loc[hit, 'event_date'], np.full(hit.sum(), 1.02),
                   marker='v', color='red', label='Observed breach')
        ax.axvline(pd.Timestamp('2023-03-10'), ls=':', color='black', label='10 March reference')
        ax.set(title=f'{coin}: SVB event study; shading = forecast-time stress flag',
               ylabel='Predicted probability', xlabel='Target event date (t+1)', ylim=(0, 1.08))
        ax.legend(fontsize=8); fig.tight_layout()
        fig.savefig(OUT / f'{coin}_SVB_probabilities.png', dpi=160)
        plt.show(); plt.close(fig)


## Event-level coverage and lead time
Choose an alert threshold using training/2022 validation only, then freeze it before examining 2023 outcomes. The threshold below is intentionally unset. Episode onset is the first observed breach in a consecutive run, not the historical date of a named crisis. Coverage means at least one alert with predictor date in [onset−L, onset−1]; lead time is onset minus the earliest alert date in that interval. Require all L forecast dates to be observed. Report event counts and false-alert days alongside coverage; a single SVB event cannot establish broad forecasting performance. Coin episodes are correlated and should not be treated as independent crises.


In [ ]:
RESULTS = ROOT / "logit/outputs/logit_decomposition"

pred_path = RESULTS / "logit_oos_predictions.csv"
VALIDATION_PATH = RESULTS / "logit_validation_2022_predictions.csv"

saved = pd.read_csv(pred_path, parse_dates=["date"])
verified = check_predictions(saved, panel)

label_mismatches = int((~verified["label_matches"]).sum())
regime_mismatches = int((~verified["regime_matches"]).sum())

print("Prediction file:", pred_path)
print("Label mismatches:", label_mismatches)
print("Regime mismatches:", regime_mismatches)

bad = (
    ~verified["label_matches"]
    | ~verified["regime_matches"]
    | verified["_merge"].ne("both")
)

if bad.any():
    display(
        verified.loc[
            bad,
            [
                "coin", "date",
                "depeg_dynamic_saved", "depeg_dynamic",
                "stress_saved", "stress", "_merge",
            ],
        ]
    )
    raise ValueError(
        "These newly loaded predictions still disagree with the audit panel. "
        "Inspect the displayed rows before calculating coverage."
    )

## Sequential 2022 OOS evidence
Import the predictions from the actual sequential validation folds; do not use fitted training probabilities or relabel a newly chosen split as the original validation. Required columns: `date,coin,depeg_dynamic,stress,p_m1_oos,p_m2_oos,p_m3_oos,fold,train_end,event_date`. `date` is the predictor date; `train_end` is the latest TRAINING OUTCOME date. It must precede every validation predictor date. Fit preprocessing on each training fold only. Preserve tuning provenance: if the same validation outcomes selected hyperparameters, their scores are development-validation evidence, not an untouched test.

The available saved OOS file starts in February 2023. Without the 2022 prediction export, this section remains incomplete. Here average precision is reported by its actual name, rather than treating it as trapezoidal PR-AUC. Single-class subsets receive no AP estimate; regime-specific results need positive and negative outcomes and multiple independent episodes for strong interpretation.


In [ ]:
RESULTS = ROOT / "logit/outputs/logit_decomposition"

pred_path = RESULTS / "logit_oos_predictions.csv"
VALIDATION_PATH = RESULTS / "logit_validation_2022_predictions.csv"
if not VALIDATION_PATH.exists():
    print('NOT COMPLETED: export original sequential 2022 validation predictions to', VALIDATION_PATH)
else:
    from sklearn.metrics import average_precision_score, brier_score_loss
    v = pd.read_csv(VALIDATION_PATH, parse_dates=['date', 'train_end', 'event_date'])
    required = ['fold', 'train_end', 'event_date', 'p_m1_oos', 'p_m2_oos']
    if not set(required).issubset(v):
        raise ValueError('Missing validation provenance/probability fields.')
    if not v.date.dt.year.eq(2022).all() or not v.train_end.lt(v.date).all():
        raise ValueError('Expected 2022 predictions with training outcomes strictly before validation.')
    if not (v.event_date - v.date).eq(pd.Timedelta(days=1)).all():
        raise ValueError('Validation event-date alignment is inconsistent.')
    # Overlapping folds require an explicit forecast-selection policy; do not average silently.
    checked = check_predictions(v.drop(columns='event_date'), panel)
    if not checked.label_matches.all() or not checked.regime_matches.all():
        raise ValueError('2022 validation labels/regimes differ from current inputs. Rerun validation.')
    checked['window'] = checked['fold'].astype(str)
    display(summary(checked))
    metrics = []
    groups = [('pooled', 'all', checked)]
    groups += [('pooled', str(s), g) for s, g in checked.groupby('stress')]
    groups += [(str(f), str(s), g) for (f, s), g in checked.groupby(['fold', 'stress'])]
    for fold, stress, g in groups:
        for col in ['p_m1_oos', 'p_m2_oos']:
            if not g[col].between(0, 1).all():
                raise ValueError('Invalid or missing probabilities.')
            metrics.append(dict(fold=fold, regime=stress, model=col, coin_days=len(g),
                positives=int(g.depeg_dynamic.sum()), prevalence=g.depeg_dynamic.mean(),
                average_precision=average_precision_score(g.depeg_dynamic, g[col])
                    if g.depeg_dynamic.nunique()==2 else np.nan,
                brier=brier_score_loss(g.depeg_dynamic, g[col])))
    metrics = pd.DataFrame(metrics)
    display(metrics)
    metrics.to_csv(OUT / '2022_sequential_oos_metrics.csv', index=False)


## Reporting limits
Report the actual analysed dates, not an assumed full-year test. A 2023 test dominated by one stressed spell cannot distinguish a general regime effect from the circumstances of one event. Use the SVB probability paths and prespecified event metrics for that window, and original sequential 2022 OOS predictions as separate evidence. Neither descriptive regime splits nor in-sample coefficient comparisons establish predictive regime dependence. Confirm sentiment release timing, dated input versions, and fold-level preprocessing before making early-warning claims.
